# Three-dimensional mesh exchange

This notebook writes and reads real VTU files through meshio. Tetrahedra, affine prisms, trilinear hexahedra and convex polyhedra retain their own geometric topology. A polygonal macroface remains one face. Finite-element order and geometric connectivity are separate contracts.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/9933d2df95b262ff3e6ec56dff73144bc2182b589bb1aa9d9c602598d5d71347/60_mesh_exchange-companion.zip"
COMPANION_SHA256 = "9933d2df95b262ff3e6ec56dff73144bc2182b589bb1aa9d9c602598d5d71347"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("foundations/geometry/60_mesh_exchange.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
from pathlib import Path
import tempfile
from io import BytesIO
from IPython.display import Image, display
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pymhm.meshes.tetrahedron import TetraMesh
from pymhm.meshes.mixed import AffineMixedMesh
from pymhm.meshes.hexahedron import HexMesh
from pymhm.meshes.hexahedron import cube_quadrature
from pymhm.meshes.polygonal import PolygonMesh
from pymhm.meshes.polyhedral import PolyhedralMesh
from pymhm.io.volume import VolumeMeshData, read_volume_mesh, write_volume_mesh

polygon = PolygonMesh([[0, 0], [1, 0], [1, 1], [0, 1], [2, 0]],
                      [[0, 1, 2, 3], [1, 4, 2]])
meshes = {
    "Tetrahedra": TetraMesh.unit_cube(),
    "Affine prisms": AffineMixedMesh.unit_cube(kind="prism"),
    "Trilinear hexahedra": HexMesh.unit_cube(2),
    "Convex polyhedra": PolyhedralMesh.extrude(polygon),
}

def cell_volumes(mesh):
    """Integrate trilinear hexahedral Jacobians or use exact native cell volumes."""
    if isinstance(mesh, HexMesh):
        points, weights = cube_quadrature(3)
        return mesh.geometry(points)[2] @ weights
    return mesh.volumes


In [ ]:
records = []
with tempfile.TemporaryDirectory(prefix="pymhm-volume-") as directory:
    for index, (name, mesh) in enumerate(meshes.items()):
        data = VolumeMeshData(mesh,
            cell_tags=np.arange(len(mesh.cells)) + 1,
            face_tags=np.arange(len(mesh.faces)) + 1,
            point_data={"position": mesh.points},
            cell_data={"volume": cell_volumes(mesh)})
        path = Path(directory) / f"geometry-{index}.vtu"
        write_volume_mesh(path, data)
        restored = read_volume_mesh(path)
        assert type(restored.mesh) is type(mesh)
        np.testing.assert_allclose(cell_volumes(restored.mesh), restored.cell_data["volume"])
        np.testing.assert_allclose(restored.point_data["position"], mesh.points)
        expected_tags = {tuple(sorted(face)): tag
                         for face, tag in zip(mesh.faces, data.face_tags, strict=True)}
        assert all(expected_tags[tuple(sorted(face))] == tag
                   for face, tag in zip(restored.mesh.faces, restored.face_tags, strict=True))
        for tag, volume in zip(restored.cell_tags, cell_volumes(restored.mesh), strict=True):
            np.testing.assert_allclose(volume, cell_volumes(mesh)[tag - 1])
        records.append((name, len(mesh.cells), len(mesh.faces), cell_volumes(mesh).sum()))
records


The last grid contains an eight-vertex cell and a six-vertex cell sharing a quadrilateral face. The checks pair each scalar field and physical ID with its actual cell after meshio groups polyhedron types. VTU blocks use increasing vertex count; files with an ambiguous noncanonical polyhedron layout are rejected.


In [ ]:
fig = plt.figure(figsize=(16, 4.2), layout="constrained")
for index, (name, mesh) in enumerate(meshes.items()):
    axis = fig.add_subplot(1, 4, index + 1, projection="3d")
    polygons = [mesh.points[face[[0, 1, 3, 2]] if isinstance(mesh, HexMesh) else face]
                for face in mesh.faces]
    axis.add_collection3d(Poly3DCollection(polygons, facecolors=(0.2, 0.55, 0.8, 0.08),
                                         edgecolors="#172534", linewidths=1.1))
    for setter, column in zip((axis.set_xlim, axis.set_ylim, axis.set_zlim), range(3), strict=True):
        setter(mesh.points[:, column].min(), mesh.points[:, column].max())
    for setter, column in zip((axis.set_xticks, axis.set_yticks, axis.set_zticks), range(3), strict=True):
        low, high = mesh.points[:, column].min(), mesh.points[:, column].max()
        setter(low + (high - low) * (np.array([0.25, 0.75]) if column == 1 else np.array([0, 0.5, 1])))
    axis.set_box_aspect(np.ptp(mesh.points, axis=0))
    axis.set(xlabel="x", ylabel="y", zlabel="z", title=name)
image = BytesIO()
fig.savefig(image, format="png", dpi=150, bbox_inches="tight", pad_inches=0.15)
plt.close(fig)
display(Image(data=image.getvalue()))


Every line above is an actual macroface edge. No cell family is converted to tetrahedra during file exchange. Geometry support does not imply that every formulation accepts every family; choose its documented mesh type.

The optional native generation checks run separately:

```bash
pixi run --locked -e meshing pytest tests/test_mesh_exchange.py tests/test_meshing_native3d.py
```

These optional contributor checks run from a source checkout. They exercise Gmsh and Netgen tetrahedral cubes, six physical boundary groups, affine Darcy patches, and meshio file round trips. The portable core imports none of these generators. Real two-rank MPI assembly and solve tests run with `pixi run --locked -e fem test-mpi`; this is a functional check, not a scalability result.
